In [0]:
# Read the values passed from the job UI FOR DEPLOYMENT
import os

def is_running_as_job():
    try:
        # If widget exists → running as job
        dbutils.widgets.get("catalog")
        return True
    except:
        return False
print("Running in Job?", is_running_as_job())
if is_running_as_job():
    catalog = dbutils.widgets.get("catalog")
    schema  = dbutils.widgets.get("schema")
    delta_table_volume = dbutils.widgets.get("delta_table_volume")
    data_ingestion_volume = dbutils.widgets.get("data_ingestion_volume")

    # delta table related paths
    delta_table_schema_store = dbutils.widgets.get("delta_table_schema_store")
    delta_table_checkpoints = dbutils.widgets.get("delta_table_checkpoints")
    delta_table_path = dbutils.widgets.get("delta_table_path")

    # data ingestion raw file related paths
    customers_data = dbutils.widgets.get("customers_data")
    orders_data = dbutils.widgets.get("orders_data")
    sales_data = dbutils.widgets.get("sales_data")

    # customer_table_name
    customers_table = dbutils.widgets.get("customers_table")

    # for deployment
    customer_delta_table_schema_path = dbutils.widgets.get("customer_delta_table_schema_path")
    customer_delta_table_checkpoint_path = dbutils.widgets.get("customer_delta_table_checkpoint_path")
    customer_delta_table_location = dbutils.widgets.get("customer_delta_table_location")
else:
    catalog = "job_orchestration"
    schema  = "default"
    delta_table_volume = "customers_volume"
    data_ingestion_volume = "job_orchestration_volume"

    # data table related paths
    delta_table_schema_store = "/Volumes/job_orchestration/default/customers_volume/schema_store"
    delta_table_checkpoints = "/Volumes/job_orchestration/default/customers_volume/checkpoints"
    delta_table_path = "/Volumes/job_orchestration/default/customers_volume/customers_table"

    # data ingestion related raw files related paths
    customers_data = "/Volumes/job_orchestration/default/job_orchestration_volume/ingest_data/customers_data/"
    orders_data = "/Volumes/job_orchestration/default/job_orchestration_volume/ingest_data/orders_data/"
    sales_data = "/Volumes/job_orchestration/default/job_orchestration_volume/ingest_data/sales_data/"

    # customer_table_name
    customers_table="customers_table"

    # for testing
    customer_delta_table_schema_path          = "/Volumes/job_orchestration/default/customers_volume/schema_store/"
    customer_delta_table_checkpoint_path      = "/Volumes/job_orchestration/default/customers_volume/checkpoints/"
    # If I pass this customer_delta_table path like this then the table created will not be registered in the unity catalog and as a result it will not be accesible by using sql commands
    # customer_delta_table_location       = "/Volumes/job_orchestration/default/customers_volume/customers_table/"
    # If i pass this customer_delta_table path like this then the table created will be registered in the unity catalog and as a result it will be accesible by suing sql commands
    customer_delta_table_location       = "job_orchestration.default.customers_table"

print("Catalog:", catalog)
print("Schema:", schema)

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{delta_table_volume};")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{data_ingestion_volume};")

Running in Job? False
Catalog: job_orchestration
Schema: default


DataFrame[]

In [0]:
%sql
-- CREATE VOLUME IF NOT EXISTS job_orchestration.default.customers_volume;

In [0]:
def mkdir_if_not_exists(path):
    try:
        # If folder exists, ls will NOT fail
        dbutils.fs.ls(path)
        print(f"Exists: {path}")
    except:
        print(f"Creating: {path}")
        dbutils.fs.mkdirs(path)

# Create all required paths safely for delta table
mkdir_if_not_exists(delta_table_schema_store)
mkdir_if_not_exists(delta_table_checkpoints)
mkdir_if_not_exists(delta_table_path)

# Create all required paths safely for data_ingestion from raw files
mkdir_if_not_exists(customers_data)
mkdir_if_not_exists(orders_data)
mkdir_if_not_exists(sales_data)

Exists: /Volumes/job_orchestration/default/customers_volume/schema_store
Exists: /Volumes/job_orchestration/default/customers_volume/checkpoints
Exists: /Volumes/job_orchestration/default/customers_volume/customers_table
Exists: /Volumes/job_orchestration/default/job_orchestration_volume/ingest_data/customers_data/
Exists: /Volumes/job_orchestration/default/job_orchestration_volume/ingest_data/orders_data/
Exists: /Volumes/job_orchestration/default/job_orchestration_volume/ingest_data/sales_data/


In [0]:
%sql 
-- LIST '/Volumes/job_orchestration/default/customers_volume'

In [0]:
spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{customers_table} USING DELTA;
""")

DataFrame[]

In [0]:
from pyspark.sql.utils import AnalysisException

# Check if csv file in customer_data location exists or not
def path_has_files(path):
    try:
        files = dbutils.fs.ls(path)
        # Filter out only real data files (csv, parquet, etc.)
        data_files = [f for f in files if f.name.lower().endswith(".csv")]
        return len(data_files) > 0
    except Exception:
        # Directory does NOT exist
        return False
auto_loader_run_status = False
if path_has_files(customers_data):
    print("✔ CSV files found — starting Auto Loader ingestion...")
    df = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", customer_delta_table_schema_path)
            .load(customers_data)
    )

    # This code creates a table that is not registered in the unity catalog and hence the generated table cannot be accessed via sql commands
    # (
    #     df.writeStream
    #         .format("delta")
    #         .option("checkpointLocation", customer_delta_table_checkpoint_path)
    #         .option("mergeSchema", "true")
    #         .outputMode("append")
    #         .trigger(availableNow=True)
    #         .start(customer_delta_table_location)
    # )

    # This code generates the table that is registered in the unity catalog hence it can be accessed via sql commands 
    (
        df.writeStream
        .option("checkpointLocation", customer_delta_table_checkpoint_path)
        .option("mergeSchema", "true")
        .outputMode("append")
        .trigger(availableNow=True)
        # .table("job_orchestration.default.customers_table")
        .table(customer_delta_table_location)
    )
    auto_loader_run_status = True
else:
    print("✘ No CSV files found — skipping ingestion.")
    auto_loader_run_status = False
    pass


✔ CSV files found — starting Auto Loader ingestion...


In [0]:
# df_loaded = spark.read.format("delta").load("/Volumes/job_orchestration/default/customers_volume/customers_table")
# display(df_loaded)

In [0]:
%sql 
-- SELECT count(*) FROM job_orchestration.default.customers_table;


#### Sending some key data to another task so that it can use it to perform tasks properly
- I want the next job to only ingest data when the customers data is successfully ingested and only the new data related to customers must be ingested in the orders_data

In [0]:
# Sending data to another task within the same job using (Task Value Propagation) method
import json

output = {
    "catalog": catalog,
    "schema": schema,
    "data_ingestion_volume":data_ingestion_volume,
    "customers_table":customers_table,
    "orders_data": orders_data,
    "sales_data":sales_data
}
if is_running_as_job():
    dbutils.jobs.taskValues.set(key="metadata", value=json.dumps(output))
else:
    display(output)

{'catalog': 'job_orchestration',
 'schema': 'default',
 'data_ingestion_volume': 'job_orchestration_volume',
 'customers_table': 'customers_table',
 'orders_data': '/Volumes/job_orchestration/default/job_orchestration_volume/ingest_data/orders_data/',
 'sales_data': '/Volumes/job_orchestration/default/job_orchestration_volume/ingest_data/sales_data/'}